# Fraud Detection in Electricity and Gas Consumption

Team project based on the Zindi challenge
[Fraud Detection in Electricity and Gas Consumption](https://zindi.africa/competitions/fraud-detection-in-electricity-and-gas-consumption-challenge).
The notebook follows the 7-stage ML blueprint:

1. Data Ingestion & Problem Framing
2. Train/Test Split & EDA Light
3. Cleaning & Feature Preprocessing
4. Baselines & Feature Selection
5. Model Exploration & CV
6. Diagnostics, Plots & Interpretability
7. Business Alignment & Turning Points

# Stage 1: Data Ingestion & Problem Framing

### Data

| File | Grain | Content |
| --- | --- | --- |
| `client_train.csv` / `client_test.csv` | one row per client | district, region, client category, creation date, **target** (train only) |
| `invoice_train.csv` / `invoice_test.csv` | one row per invoice | date, tariff, counter info, consumption levels 1–4, meter indices, counter type (ELEC/GAZ) |

The Zindi test set has no labels: it is only used for the submission, we evaluate on our own holdout.

## 1.1 Loading and overview the data (first look)

In [ ]:
# Imports
from pathlib import Path
from IPython.display import display
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# Load the raw tables
client_train = pd.read_csv("data/train/client_train.csv")
invoice_train = pd.read_csv("data/train/invoice_train.csv", low_memory=False)
client_test = pd.read_csv("data/test/client_test.csv")
invoice_test = pd.read_csv("data/test/invoice_test.csv", low_memory=False)

for name, df in {
    "client_train": client_train,
    "invoice_train": invoice_train,
    "client_test": client_test,
    "invoice_test": invoice_test,
}.items():
    print(f"{name:14s} {df.shape}")


In [ ]:
# First look: client tables
display(client_train.head())
display(client_test.head(10))
display(client_train.describe().T)
# print(client_train["target"].value_counts().sum())
print("This is the fraud percantage")
print(client_train["target"].mean()*100)

In [ ]:
# First look: invoice table
display(invoice_train.head(21))

In [ ]:
# Missing and zero values
print("Do we have NaN-values?: ")
print(invoice_train.isna().sum())

print("\nDo we have Values, that are 'zero'?:")
display((invoice_train == 0).sum())

## 1.2 Column overview

In [ ]:
# Column overview: client table
def overview(df):
    return pd.DataFrame({
        "dtype": df.dtypes,
        "n_unique": df.nunique(),
        "n_missing": df.isna().sum(),
        "example": df.iloc[0],
    })


overview(client_train)


In [ ]:
# Column overview: invoice table
overview(invoice_train)

## 1.3 Join key and data grain

### What does it mean?

- **Data grain** = what one row stands for: in `client_train` one row is **one client**, in `invoice_train` one row is **one invoice**.
- **Join key** = the column that links both tables: `client_id`.
- One client has **many** invoices (1 : n). So before joining, the invoices must be summarised per client (one row per client). Otherwise every client would appear ~33 times, together with their `target`.

In [ ]:
# Join key and data grain
print("Each client appears only once:", client_train["client_id"].is_unique)
print("Invoices without a known client:", (~invoice_train["client_id"].isin(client_train["client_id"])).sum())
print("Clients without any invoice:", (~client_train["client_id"].isin(invoice_train["client_id"].unique())).sum())

# How many invoices does one client have?
invoice_train.groupby("client_id").size().describe()

## 1.4 Target distribution

In [ ]:
# Target distribution
print(client_train["target"].value_counts(dropna=False))
print()
print(client_train["target"].value_counts(normalize=True).round(3)*100)

client_train["target"].value_counts().plot.bar(title="Target distribution (0 = no fraud, 1 = fraud)")
plt.show()


# Stage 1 End: Results & Learnings (Marker)

## 1.5 Problem Framing

### Stakeholder, goal and metric

**Stakeholder:** STEG (Tunisian Company of Electricity and Gas), revenue protection / inspection teams.

**Problem:** Some customers manipulate their electricity or gas meters, which causes large revenue losses.
On-site inspections are expensive and capacity is limited, so STEG cannot check every customer.

**Goal:** Rank customers by their probability of fraud, based on their client data and billing history,
so inspection teams visit the most suspicious customers first.

**ML task:** Binary classification at client level.
- Target `y`: `target` in `client_train.csv` (1 = fraud, 0 = no fraud), one label per client
- Signal: ~4.5 M invoices in `invoice_train.csv`, aggregated per `client_id`
- Class imbalance: 5.6 % fraud

**Metrics:**
- **Zindi:** ROC-AUC (competition metric). It measures how well the model ranks fraud above non-fraud, without a threshold.
- **Business:** Recall & F2 at the chosen threshold. A missed fraud case means lost revenue, so recall matters most;
  F2 weights recall higher but still punishes flagging everyone as fraud.
- Accuracy is useless here: predicting "never fraud" already reaches 94.4 % and finds no fraud at all.

### Rubber-duck checklist

- [x] **Do we know exactly what the target $y$ is?** → Yes: `target` in `client_train.csv`, one label per client.
- [x] **Regression, classification, clustering or anomaly detection?** → Binary classification, output as fraud probability for ranking.
- [x] **Do all features exist at prediction time?** → Not fully verifiable: the data has no fraud-detection date. Invoices issued after detection (counter replacement, corrections) could leak the label. Risk noted; check in Stage 2 (end of invoice history, fraud vs. non-fraud) and Stage 6 (suspiciously dominant features).

## 1.6 Learnings (Marker)

### Stage 1: Marker – Problem type first, then algorithm and metric

Three separate levels:

1. **Problem type:** what should come out? Decided only by what $y$ looks like, and whether there is a $y$ at all.
2. **Algorithm:** what do I compute with? (e.g. Decision Tree, LightGBM)
3. **Metric:** how do I measure if it is good? (e.g. Accuracy, Recall, RMSE)

| Problem type | $y$ looks like | Question | Example metric | Example algorithm |
| --- | --- | --- | --- | --- |
| **Regression** | a number, e.g. 1,234 kWh | How much? | RMSE, MAE | Linear Regression, Decision Tree |
| **Classification** | a class, e.g. 0/1 | Which group? | Accuracy, Recall, Precision, F1, AUC | Logistic Regression, Decision Tree |
| **Clustering** | **no $y$** | Which clients are similar? | Silhouette score | k-Means |
| **Anomaly detection** | **no $y$** (or almost none) | Who is unusual? | Precision@k | Isolation Forest |

- Accuracy is a classification metric; regression measures the distance to the true number (RMSE).
- A Decision Tree is an algorithm, not a problem type: it does regression and classification, not clustering.
- Recall and Precision count false positives/negatives, so they need the true answer (labels).
- Fraud is often an anomaly-detection problem, namely when nobody knows which clients committed fraud.

# Stage 2: Data Splitting & EDA Light

### Split strategy

- **What is split:** the clients in `client_train` (one row = one client = one `target`). The invoices follow their client via `client_id`, so all invoices of a client stay on the same side.
- **Why not split the invoices:** the same client (and their `target`) would end up in train and holdout → data leakage.
- **Stratified:** `stratify=y` keeps the fraud rate (5.6 %) equal in train and holdout.
- **No time-based split:** the `target` has no date (one label per client), so there is no "past vs. future" of clients.
- **Holdout (20 %)** stays untouched until the final evaluation in Stage 7. All EDA below uses the train part only.

## 2.1 Train/holdout split at client level

In [ ]:
# Stratified train/holdout split at client level
from sklearn.model_selection import train_test_split

train_clients, holdout_clients = train_test_split(
    client_train,
    test_size=0.2,
    stratify=client_train["target"],
    random_state=42,
)

print("train  :", train_clients.shape, f"fraud rate {train_clients['target'].mean():.2%}")
print("holdout:", holdout_clients.shape, f"fraud rate {holdout_clients['target'].mean():.2%}")

In [ ]:
# Invoices follow their client
train_invoices = invoice_train[invoice_train["client_id"].isin(train_clients["client_id"])]
holdout_invoices = invoice_train[invoice_train["client_id"].isin(holdout_clients["client_id"])]

print("train invoices  :", train_invoices.shape)
print("holdout invoices:", holdout_invoices.shape)
print("clients on both sides:", len(set(train_invoices["client_id"]) & set(holdout_invoices["client_id"])))

In [ ]:
# Save holdout client IDs so the whole team uses the same split
Path("data/split").mkdir(exist_ok=True)
holdout_clients[["client_id"]].to_csv("data/split/holdout_ids.csv", index=False)

In [ ]:
#create merged training dataset for when every columns is needed during analysis
train_merged = train_invoices.merge(train_clients, on="client_id", how="inner")

## 2.2 EDA and cleaning

### 2.2.1 Leakage check: end of the invoice history

#### Why this check?

The data has no fraud-detection date (Stage 1, checklist). If STEG replaces the counter or corrects invoices **after** detecting a fraud, fraud clients look different at the **end** of their history.
A large difference is a warning sign for leakage; a small one does not prove there is none.

In [ ]:
# Leakage check: do fraud clients look different at the end of their history? (train only)
inv = train_invoices.copy()
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])

per_client = inv.groupby("client_id").agg(
    last_invoice=("invoice_date", "max"),
    n_counters=("counter_number", "nunique"),
).join(train_clients.set_index("client_id")["target"])

per_client.groupby("target").agg(
    median_last_invoice=("last_invoice", "median"),
    mean_n_counters=("n_counters", "mean"),
)

### 2.2.2 Feature investigation and cleaning

#### counter_statue

##### Analysis

`counter_statue` has a few invalid values, should be only between 0 and 5.

The amount is quite low but let's check first if it is somehow related to the target outcome first.

Do the decision on training data.

In [ ]:
valid_values = ["0", "1", "2", "3", "4", "5"]
invalid_invoice = train_invoices[~train_invoices["counter_statue"].isin(valid_values)]

print(f"Number of invalid rows {invalid_invoice.shape[0]}")
merged_invalid_rows = pd.merge(train_clients, invalid_invoice, on="client_id", how="inner")


individual_clients = merged_invalid_rows["client_id"].value_counts().count()
fraudulent = merged_invalid_rows["client_id"][merged_invalid_rows["target"] != 0].count()

print(f"By deleting rows with invalid 'counter_statue' we lose data from {individual_clients} individual clients and {fraudulent} fraudulent entries")



##### Conclusion

It is probably safe to just delete those columns since the impact is minor and does not further delete positive classes.

Delete in both splits.

In [ ]:
train_invoices = train_invoices[train_invoices["counter_statue"].isin(valid_values)]
holdout_invoices = holdout_invoices[holdout_invoices["counter_statue"].isin(valid_values)]
train_merged = train_merged[train_merged["counter_statue"].isin(valid_values)]

train_invoices["counter_statue"].value_counts()

#### months_number

##### Analysis

Does this reporesent the 12 months?

In [ ]:
train_invoices["months_number"].value_counts().head(20)

In [ ]:
rows_with_invalid_months = train_invoices[~train_invoices["months_number"].between(1,12)]
invalid_months = rows_with_invalid_months["months_number"].value_counts()
invalid_months_ratio = 100 * rows_with_invalid_months.size / train_invoices.shape[0]
print(f"There are  {rows_with_invalid_months.size} invalid month entries, ratio: {invalid_months_ratio:.2f} %\n")
print(invalid_months)

In [ ]:
tmp = pd.concat([pd.to_datetime(train_invoices["invoice_date"]).dt.month, train_invoices["months_number"]], axis=1)
display(tmp.head())
tmp[["invoice_date", "months_number"]].corr()

Probably not the month of the invoice.

Let's see if it is the number of months since the last invoice, so the number of months in which the consumption occurred.

In [ ]:
# Months between consecutive invoices of the same client (train only)
invoice_gaps = train_merged.copy()
correlation_matrix = invoice_gaps[["months_number", "target"]].corr()
invoice_gaps["invoice_date"] = pd.to_datetime(invoice_gaps["invoice_date"])
invoice_gaps = invoice_gaps.sort_values(["client_id", "invoice_date"]).reset_index(drop=True)

# diff() within each client -> first invoice of every client gets NaT (no previous row)
# do it raw nad also sort by counter_type (GAZ/ELEC)
days_since_last_raw = invoice_gaps.groupby(["client_id"])["invoice_date"].diff().dt.days
days_since_last_sort = invoice_gaps.groupby(["client_id", "counter_type"])["invoice_date"].diff().dt.days
invoice_gaps["months_since_last_raw"] = (days_since_last_raw / 30.44).round()
invoice_gaps["months_since_last_sort"] = (days_since_last_sort / 30.44).round()

# Keep only rows where the difference was calculated (drops each client's first invoice)
invoice_gaps = invoice_gaps.dropna(subset=["months_since_last_raw", "months_since_last_sort"])
invoice_gaps["months_since_last_raw"] = invoice_gaps["months_since_last_raw"].astype(int)
invoice_gaps["months_since_last_sort"] = invoice_gaps["months_since_last_sort"].astype(int)

difference_raw = np.absolute(invoice_gaps["months_since_last_raw"] - invoice_gaps["months_number"])
difference_sort = np.absolute(invoice_gaps["months_since_last_sort"] - invoice_gaps["months_number"])

print(correlation_matrix)

In [ ]:
difference_raw = np.absolute(invoice_gaps["months_since_last_raw"] - invoice_gaps["months_number"])
difference_sort = np.absolute(invoice_gaps["months_since_last_sort"] - invoice_gaps["months_number"])
ratio_raw = 100 * (difference_raw <= 1).sum() / difference_raw.size
ratio_sort = 100 * (difference_sort <= 1).sum() / difference_sort.size
print("Ratio of rows where months_number is the time since the last invoice +/- 1 month:")
print(f"Sorted only by client {ratio_raw:.2f}")
print(f"Sorted also by GAZ/ELEC {ratio_sort:.2f}")

##### Conclusion

It is not clear what this columns says, since
1. The top entries are not nicely in the 1 - 12 range
2. No correlation with the invoice date
3. No strong evidence that it is the months since the last invoice (duration of consumption)

#### Consumption levels, new and old index

##### Analysis

Let's see how consumption and counter indices are related

In [ ]:
# is index difference same as sum of consumptions?
sum_of_conumption = train_invoices[[f"consommation_level_{i}" for i in range(1, 5)]].sum(axis=1)

mismatch = sum_of_conumption != (train_invoices["new_index"] - train_invoices["old_index"])
mismatch_ratio = 100 * mismatch.sum() / train_invoices.shape[0]

print(f"The mismatch ratio is {mismatch_ratio:.2f} %")

In [ ]:
# how high is fraudulence in the mismatch group
# get client_id where mismatch happened
mismatched_clients = train_invoices["client_id"][mismatch].unique()
# sum up the frauds 
sum_of_frauds = train_clients["target"][train_clients["client_id"].isin(mismatched_clients)].sum()

# calculate the fraudulence ratio between all clients with mismatches
fraud_ratio_mismatch = 100 * sum_of_frauds / mismatched_clients.size
print(f"The group that had mismatches has a fraudulence rate of {fraud_ratio_mismatch:.2f} %")

In [ ]:
# are the consumption levels mutually exclusive as expected?
# make a sample so the plot is quicker. The sample should keep same fraudulence ratio
plot_sample = train_merged.groupby("target", group_keys=False).sample(frac=0.1)

levels = [f"consommation_level_{i}" for i in range(1, 5)]
# still takes ~3m, so uncomment if neccessary
#sns.pairplot(plot_sample[levels + ["target"]], hue="target", vars=levels, plot_kws={"alpha": 0.3})

##### Conclusion

In the client group where sum of consumptions does not match the difference of the index the fraudulence ratio is higher, so it seems to be a good idea to create a "mismatch" feature to capture this.

The consuption levels are not mutually exclusive.

Fraudulence seems to be prevalent in all the levels and in the lower value range. But the ranges are not the same, so we cannot simply add the values.

There are strong outliers in the levels, so some kind of preprocessing should be done when using reggression models.

#### Counter_type

##### Analysis

Is the fraudlunce going on predominantly on Gas or Electricity?

In [ ]:
train_temp = train_merged[["target", "counter_type"]]
train_temp["counter_type"] = train_temp["counter_type"].map({"ELEC":0,"GAZ":1})
train_temp.corr()

##### Conclusion

No correlation, so fraud is going on on both.

### Disrict

#### Analysis

Let's see if and how it is related to region.

In [ ]:
districts_by_region = (
    train_clients.groupby("disrict")["region"]
    .unique()
    .apply(sorted)
)
display(districts_by_region)


In [ ]:
counts = train_clients.groupby(["region", "target"]).size().unstack(fill_value=0)
fraud_ratio = counts[1] / counts[0]

fig, ax = plt.subplots(figsize=(10, 5))
fraud_ratio.plot(kind="bar", ax=ax)
ax.set_xlabel("Region")
ax.set_ylabel("Frauds / non-frauds")
ax.set_title("Fraud to non-fraud ratio per region")
plt.tight_layout()
plt.show()

In [ ]:
counts = train_clients.groupby(["disrict", "target"]).size().unstack(fill_value=0)
fraud_ratio = counts[1] / counts[0]

fig, ax = plt.subplots(figsize=(10, 5))
fraud_ratio.plot(kind="bar", ax=ax)
ax.set_xlabel("District")
ax.set_ylabel("Frauds / non-frauds")
ax.set_title("Fraud to non-fraud ratio per destrict")
plt.tight_layout()
plt.show()

##### Conclusion

Regions (with a few examples) seem to belong to one specific district.

This means Region is a more granular separation of location.

There are areas where fraud is double than others.

It has to be disussed if it is ethically acceptable to profile people to where they come from.

# Stage 2 End: Results & Learnings (Marker)

## 2.3 Split decisions

### Rubber-duck checklist

- [x] **Was the test set split off before any deeper analysis/scaling?** → Yes: 20 % holdout of `client_train`, split at client level before EDA.
- [x] **Is there a time dimension in the data?** → Invoices have dates, but the `target` has none (one label per client) → stratified random split at client level is valid.
- [x] **Is the target class extremely imbalanced?** → Yes, 5.6 % fraud → `stratify=y`.

## 2.4 Learnings (Marker)

Client_Train___    
INT|'disrict' - DROP//replace name "district" 
INT|'client_id', - DROP// 
INT|'client_catg',  - // -- dummyvariables -- feature
INT|'region', - //correlation matrix -- dummyvariables -- feature
DATE|'creation_date' DROP//
INT|'target' -- //drop and use it as "y"

Invoice_Train____
client_id' DROP//
invoice_date' DROP//
tarif_type' //correlation matrix
counter_number' DROP//
counter_statue' //correlation matrix (DROP//invalid rows 47!!)
counter_code' //correlation matrix
reading_remarque' //definitly a feature -- [drop rows over 100]
counter_coefficient' //sound like a feature
consommation_level_1'// keep as feature
consommation_level_2'// keep as feature
consommation_level_3'// keep as feature
consommation_level_4'// keep as feature
old_index' -- DROP// combine (consumption_missmatch = sum(consumLVL1-4) != dif(old_index, new_index)   )
new_index' -- DROP// 
months_number' DROP//
counter_type' // feature -- dummyvariable



Issue "Fraud Detection at 1 client over a bigger time period" - (RandomSelection)
Pipeline Structure

In [ ]:
# Is counter_type a useful feature? Fraud rate per type combination (train only)
types_per_client = (
    train_invoices.groupby("client_id")["counter_type"]
    .agg(lambda s: "+".join(sorted(s.unique())))   # e.g. "ELEC", "GAZ", "ELEC+GAZ"
    .rename("types")
    .to_frame()
    .join(train_clients.set_index("client_id")["target"])
)

types_per_client.groupby("types")["target"].agg(n_clients="count", fraud_rate="mean")

In [ ]:
# Correlation matrix of numeric client-level features (train only)
corr = per_client.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix (train, client level)")
plt.show()

In [ ]:
# Check: does the sum of the 4 levels equal new_index - old_index? (train only)
levels = ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]
total = train_invoices[levels].sum(axis=1)
index_diff = train_invoices["new_index"] - train_invoices["old_index"]

print("share of invoices where sum == index difference:", (total == index_diff).mean().round(3))

In [ ]:
# Mismatch between levels and index difference: more common for fraud? (train only)
mismatch = train_invoices.assign(mismatch=(total != index_diff))

mismatch_per_client = (
    mismatch.groupby("client_id")["mismatch"].any()
    .to_frame()
    .join(train_clients.set_index("client_id")["target"])
)

print(mismatch_per_client.groupby("target")["mismatch"].mean().round(3))

In [ ]:
invoice_train.describe().T

In [ ]:
# Consumption levels: overview (train only)
levels = ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]

summary = train_invoices[levels].describe(percentiles=[0.5, 0.9, 0.99]).T
summary["share_zero"] = (train_invoices[levels] == 0).mean()
summary

In [ ]:
# Distribution of consumption level 1 (log scale, train only)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_invoices["consommation_level_1"].plot.hist(bins=100, ax=axes[0], title="Level 1 (raw)")
np.log1p(train_invoices["consommation_level_1"]).plot.hist(bins=100, ax=axes[1], title="Level 1 (log1p)")
plt.show()

In [ ]:
# Mean consumption per client, fraud vs. no fraud (train only)
per_client = (
    train_invoices.groupby("client_id")[levels].mean()
    .join(train_clients.set_index("client_id")["target"])
)

display(per_client.groupby("target")[levels].median())

sns.boxplot(data=per_client, x="target", y=np.log1p(per_client["consommation_level_1"]))
plt.title("Mean consumption level 1 per client (log1p) by target")
plt.show()

In [ ]:
# reading_remarque: how often, and how much fraud per code? (train only)
remarque = (
    train_invoices[["client_id", "reading_remarque"]]
    .merge(train_clients[["client_id", "target"]], on="client_id")
    .groupby("reading_remarque")["target"]
    .agg(n_invoices="count", fraud_rate="mean")
    .sort_values("n_invoices", ascending=False)
)
remarque

In [ ]:
invoice_train["reading_remarque"].value_counts()
# invoice_test["reading_remarque"].value_counts()

In [ ]:
client_train.groupby("disrict")["region"].unique().apply(sorted)

In [ ]:
# Regions that appear in only one of the two files
train_regions = set(client_train["region"])
test_regions = set(client_test["region"])

print("only in train:", sorted(train_regions - test_regions))
print("only in test :", sorted(test_regions - train_regions))

In [ ]:
print(client_train["disrict"].value_counts())

In [ ]:
print(client_train["region"].value_counts())
print(client_test["region"].value_counts())
# print(invoice_train.duplicated().sum())     #drop duplicated rows
